# Bentchmark cases

Example of creation of a multifidelity Neural Network to create the forward model.

In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from hyperopt.pyll.base import scope
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.optimizers import Adam,Nadam,Adamax
from time import perf_counter
import tensorflow as tf

import tinyDA as tda
from scipy.stats import multivariate_normal, uniform
import arviz as az
import keras


from module_utils import * 
sys.path.append('../utils')
from Structure import *

from pathlib import Path

# path to the current notebook
current_file_path = Path().resolve()
# path to the current folder
load_context_functions(current_file_path.parent.name)



# fixing the seeds for reproducibility purposes
seed = 10
tf.random.set_seed(seed)
np.random.seed(seed)
keras.utils.set_random_seed(seed)

## Creation of the model 
Here the functions to be approximated are introduced

In [ ]:
cases={"Basic", "Discontinuous", "Oscillatory"}
# insert the case to consider 

example="Basic"

match example:
    case "Basic":
        # high fidelity and low fidelity functions
        highfid = lambda x: (6.*x/5-2.)**2*np.sin(12.*x/5-4.)
        lowfid  = lambda x: 0.5*highfid(x) + 10*(x/5-0.5) + 5.
        # number of training values for high and low fidelity
        Nhf = 5
        Nlf = 32
        N = Nhf + Nlf
        # test set
        xhf = np.linspace(0,5,Nhf)
        xlf = np.linspace(0,5,Nlf)
        x_test = np.linspace(0,5,1000)
        Yhf = highfid(xhf)
        Ylf = lowfid(xlf)
        # epochs
        NepoLF = 1000
        NepoHF = 500
        
    case "Discontinuous":

        lowfid = lambda x: (0.5*(6.*x/5.-2.)**2*np.sin(12.*x/5.-4)+10.*(x/5-0.5)-5.)*(x<2.5) + (3+0.5*(6.*x/5.-2)**2*np.sin(12.*x/5.-4)+10*(x/5.-0.5)-5.)*(x>2.5)
        highfid = lambda x: (2*lowfid(x)- 20*x/5.+20)*(x/5.<0.5) + (4+2*lowfid(x)- 20*x/5.+20)*(x/5.>0.5)
       
        Nhf = 6
        Nlf = 32
        N = Nhf + Nlf
        
        xhf = np.linspace(0,5,Nhf)
        xlf = np.linspace(0,5,Nlf)
        x_test = np.linspace(0,5,1000)
        Yhf = highfid(xhf)
        Ylf = lowfid(xlf)
        
        NepoLF = 5000
        NepoHF = 1200

    case "Oscillatory":

        lowfid = lambda x: np.sin(8*np.pi*x/5)
        highfid = lambda x: (x/5-np.sqrt(2))*lowfid(x)**2
        
        Nhf = 15
        Nlf = 64
        N = Nhf + Nlf
        
        xhf = np.linspace(0,5,Nhf)
        xlf = np.linspace(0,5,Nlf)
        x_test = np.linspace(0,5,1000)
        Yhf = highfid(xhf)
        Ylf = lowfid(xlf)
        
        NepoLF = 3000
        NepoHF = 3000

    case _:
        raise ValueError(f"Example {example} not recognized")




In [ ]:
#%% Plot functions
plt.figure()
plt.plot(x_test,highfid(x_test),'r', label = 'high-fidelity sol')
plt.plot(x_test,lowfid(x_test),'g', label = 'low-fidelity sol')
plt.plot(xhf,highfid(xhf),'ro', label = 'high-fidelity data', markersize = 4)
plt.plot(xlf,lowfid(xlf),'go', label = 'low-fidelity data', markersize = 4)
plt.xlabel('x')
plt.legend()
plt.title('Benchmark 1D')
plt.show()

## Low fidelity Neural Network

In [ ]:
bestLF_params = {'lr' : 0.0255, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam'}
modelLF=NetworkFactory.build_network('LF',params=bestLF_params,data_train=xlf,output_train=Ylf,N=NepoLF,n=Nlf,train=True,do_HPO=False,verbose=False)
yLF=modelLF.prediction(x_test)
(mse_LF,R_LF)=modelLF.performance(x_test,highfid(x_test))

## High fidelity Neural Network

In [ ]:
bestHF_params = {'lr' : 0.0255, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam'}
modelHF=NetworkFactory.build_network('LF',params=bestHF_params,data_train=xhf,output_train=Yhf,N=NepoHF,n=Nhf,train=True,do_HPO=False,verbose=False)
yHF=modelHF.prediction(x_test)
(mse_HF,R_HF)=modelHF.performance(x_test,highfid(x_test))

# 2 step model 

In [ ]:
bestLF_params = {'lr' : 0.0255, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam'}
best_params = {'kernel_init': 'uniform', 'l2weight': 0.00010018625799978436, 'lr': 0.07093837044166487, 'nodes': 74.0, 'opt': 'Adamax'}
N=[NepoLF,NepoHF]
n=[Nlf,Nhf]
names=['LF','HF']

params=[bestLF_params,best_params]
final_model=NetworkFactory.build_network("2step",names,params=params,data_train=[xlf,xhf],output_train=[Ylf,Yhf],N=N,n=n,do_HPO=False,verbose=False)
y_test=final_model.prediction(x_test)

input_HF = np.concatenate(
                    (x_test.reshape(-1,1),  final_model.model_list[0].prediction(x_test).reshape(-1,1)),axis=1
                ) 
(mse_MF,R_MF)=final_model.model_list[1].performance(input_HF,highfid(x_test))

### Plot of the predicted models

In [ ]:
#LF single-fidelity
plt.figure()
plt.plot(xlf,Ylf,'go', label = 'LF training data', markersize = 4)
plt.plot(x_test,lowfid(x_test),'g', label = 'exact LF') 
plt.plot(x_test,yLF,'k--', label = 'pred LF')
plt.legend()
plt.title('Low fidelity model')

#HF single-fidelity
plt.figure()
plt.plot(xhf,Yhf,'ro', label = 'HF training data')
plt.plot(x_test,highfid(x_test),'-r', label = 'exact HF')
plt.plot(x_test,yHF,'--k', label = 'pred HF')
plt.legend()
plt.title('High fidelity model - single-fidelity')

#2-step MF
plt.figure()
plt.plot(xhf,Yhf,'ro', label = 'HF training data')
plt.plot(x_test,highfid(x_test),'-r', label = 'exact HF')
plt.plot(x_test,y_test,'--k', label = 'pred HF')
plt.legend()
plt.title('High fidelity model - 2-step')